# EDA

## MGF summary

Per-file summary of every MGF under `data_mgf/`:

- **spectra** — row count (one per `BEGIN IONS … END IONS` block)
- **peaks_med** — median peaks per spectrum
- **mz_med** — median precursor m/z

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

MGF_ROOTS = [Path('data_mgf/ecoli'), Path('data_mgf/wastewater')]
PEAK_RE   = re.compile(r'^\s*[0-9.eE+-]+\s+[0-9.eE+-]+\s*$')

def summarize_mgf(path: Path) -> dict:
    spectra = 0
    peak_counts: list[int] = []
    pep_mz:      list[float] = []
    cur_peaks = 0

    with open(path) as f:
        for raw in f:
            s = raw.rstrip('\n')
            if s.startswith('BEGIN IONS'):
                spectra += 1
                cur_peaks = 0
            elif s.startswith('END IONS'):
                peak_counts.append(cur_peaks)
            elif s.startswith('PEPMASS='):
                tok = s[len('PEPMASS='):].split()
                try:
                    pep_mz.append(float(tok[0]))
                except (ValueError, IndexError):
                    pass
            elif PEAK_RE.match(s):
                cur_peaks += 1

    sample = re.sub(r'_[12]\.mgf$', '', path.name)
    return {
        'sample':    sample,
        'file':      path.name,
        'spectra':   spectra,
        'peaks_med': int(np.median(peak_counts))         if peak_counts else 0,
        'mz_med':    round(float(np.median(pep_mz)), 2)  if pep_mz      else None,
    }

rows = []
for root in MGF_ROOTS:
    for f in sorted(root.glob('*.mgf')):
        rows.append(summarize_mgf(f))
pd.DataFrame(rows)

,sample,file,spectra,peaks_med,mz_med
0,Ecoli_EV,Ecoli_EV_1.mgf,10881,1131,665.86
1,Ecoli_EV,Ecoli_EV_2.mgf,10924,1100,668.93
2,wastewater_Sample1,wastewater_Sample1_1.mgf,17100,334,609.78
3,wastewater_Sample1,wastewater_Sample1_2.mgf,17600,340,605.78
4,wastewater_Sample2,wastewater_Sample2_1.mgf,16144,334,592.87
5,wastewater_Sample2,wastewater_Sample2_2.mgf,10306,291,566.31


## Ground truth summary

Per-sample summary read from the MaxQuant database-search xlsx exports under `data/ecoli/`:

- **Total PSMs** — all rows in the search export
- **PEP<0.01** — PSMs passing MaxQuant's per-PSM confidence cutoff (this is what `build_groundtruth.py` writes to `ground_truth/`)
- **Unique Peptides** — distinct bare-AA sequences in the PEP<0.01 subset

In [2]:
XLSX_FILES = {
    'Ecoli_EV_1': Path('data/ecoli/Database_search_output_Ecoli_EV_1.xlsx'),
    'Ecoli_EV_2': Path('data/ecoli/Database_search_output_Ecoli_EV_2.xlsx'),
}

def summarize_gt(name: str, path: Path) -> dict:
    df = pd.read_excel(path)
    confident = df[df['PEP'] <= 0.01]
    return {
        'File':            name,
        'Total PSMs':      len(df),
        'PEP<0.01':        len(confident),
        'Unique Peptides': confident['Sequence'].nunique() if 'Sequence' in confident else None,
    }

pd.DataFrame([summarize_gt(n, p) for n, p in XLSX_FILES.items()])

,File,Total PSMs,PEP<0.01,Unique Peptides
0,Ecoli_EV_1,1495,1190,674
1,Ecoli_EV_2,1667,1329,745
